# Tahap 6 — Pelaporan

**Tujuan**: menyajikan hasil sedemikian rupa sehingga setiap klaim dapat ditelusuri ke
eksperimen yang mengisolasinya.

**Prasyarat**: Tahap 0-5 selesai (minimal jalur minimum: 0→1→2→3→4.1→5.1).

**Navigasi Eksekusi_RL**: [Tahap 0](00_Bekukan_Substrat.ipynb) → [Tahap 1](01_Tetapkan_Rezim.ipynb) → [Tahap 2](02_Replikasi_Baseline_PDQN.ipynb) → [Tahap 3](03_Eksperimen_Pivot.ipynb) → [Tahap 4](04_Solusi_RRM_Arsitektur.ipynb) → [Tahap 5](05_Robustness_Stabilitas.ipynb) → **[Tahap 6]**

Dokumen rujukan: `../Dokumen_Penting/Rencana_Eksekusi_Penelitian.md` (rencana lengkap), `../Dokumen_Penting/Rumusan_Masalah_Teknis_RL.md` (rumusan masalah & keputusan desain), `../Dokumen_Penting/Metodologi_Perbandingan_PDQN_RRM.md` (aturan atribusi).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath(".."))
import common
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.precision", 4)
ROOT = common.ROOT
print("ROOT:", ROOT)
print("Substrat saat ini:", common.SUBSTRAT)

## 6.1 Muat seluruh hasil tahap sebelumnya

In [ ]:
import glob
outputs = {}
for f in sorted(glob.glob(os.path.join(common.OUTDIR, "*.json"))):
    name = os.path.basename(f).replace(".json", "")
    with open(f, encoding="utf-8") as fh:
        outputs[name] = json.load(fh)
print("Berkas hasil tersedia:")
for k in outputs:
    print(" -", k)

## 6.2 Tangga ablasi lengkap (isi manual berdasar Tahap 2-5)

In [ ]:
tangga = pd.DataFrame([
    dict(titik="A",  deskripsi="PDQN, trust statis (0,5)", gini_mean=np.nan, keterangan="Tahap 2"),
    dict(titik="A'", deskripsi="PDQN, trust statis (level akhir B)", gini_mean=np.nan, keterangan="Tahap 3 (kontrol)"),
    dict(titik="B",  deskripsi="PDQN, trust dinamis", gini_mean=np.nan, keterangan="Tahap 3 (pivot)"),
    dict(titik="E",  deskripsi="B + RRM (arsitektur identik)", gini_mean=np.nan, keterangan="Tahap 4"),
    dict(titik="C",  deskripsi="B + arsitektur baru (tanpa RRM)", gini_mean=np.nan, keterangan="Tahap 4 (opsional)"),
    dict(titik="D",  deskripsi="C + RRM", gini_mean=np.nan, keterangan="Tahap 4 (opsional)"),
])
print("[TODO] Isi kolom gini_mean dari hasil Tahap 2-4 tersimpan.")
display(tangga)

# selisih ter-atribusi
print("\nSelisih ter-atribusi (isi setelah tangga di atas lengkap):")
print("B-A   = kerusakan akibat performativitas")
print("E-B   = kontribusi RRM berdiri sendiri")
print("C-B   = kontribusi arsitektur (bila dijalankan)")
print("D-C   = kontribusi RRM di atas arsitektur baru (bila dijalankan)")

## 6.3 Kurva performativitas vs beban (dari Tahap 1)

In [ ]:
if "01_performativity_sweep" in outputs:
    df_perf = pd.DataFrame(outputs["01_performativity_sweep"])
    display(df_perf.groupby("load_multiplier")["trust_mean"].agg(["mean", "std"]))
else:
    print("[belum tersedia -- jalankan Tahap 1]")

## 6.4 Tabel baseline lengkap S0-S3

In [ ]:
if "01_baseline_S0-S3" in outputs:
    display(pd.DataFrame(outputs["01_baseline_S0-S3"]))
else:
    print("[belum tersedia -- jalankan Tahap 1.3]")

## 6.5 Checklist kejujuran wajib (§6.2 Rencana Eksekusi)

- [ ] **PDQN kalah dari `greedy_util` bahkan pada trust statis** (p=0,0039, preseden arsip)
      dinyatakan eksplisit, bukan disembunyikan
- [ ] **Sebagian masalah yang ditemukan bukan kelemahan PDQN**, melainkan cacat formulasi
      reward/lingkungan yang diperbaiki di substrat (Tahap 0) dan diberikan juga kepada PDQN
- [ ] **Rezim eksperimen dipilih berdasarkan pengukuran** (Tahap 1), bukan intuisi — kurva
      §6.3 dilampirkan sebagai bukti
- [ ] Keterbatasan simulator yang relevan (`LAPORAN_VALIDASI.md` §V5.3–V5.4) dikutip,
      terutama bahwa klaim tentang dinamika trust pada beban kanonik tidak didukung
- [ ] Anggaran pelatihan & jumlah konfigurasi yang dicoba **per lengan** dicantumkan
      (bukti kesetaraan, Lapisan 2 metodologi)
- [ ] Seluruh uji statistik menyertakan ukuran efek, bukan hanya p-value

## 6.6 Batas klaim (format `LAPORAN_VALIDASI.md` §V5.4)

Isi manual, contoh kerangka:

> Hasil ini **mendukung** klaim bahwa performativitas trust menurunkan performa PDQN pada
> rezim beban [...], dan bahwa RRM memulihkan [...]% dari kerusakan tersebut, pada substrat
> Klaster 12 dengan konfigurasi reward yang telah diseimbangkan (Tahap 0).
>
> Hasil ini **tidak mendukung** klaim serupa pada beban kanonik (1×), di mana performativitas
> terbukti tidak material.
>
> Hasil ini **tidak mendukung** [...] karena [...].

In [ ]:
print("Isi §6.6 secara manual di sel markdown di atas setelah seluruh tahap selesai.")
print("Ekspor notebook ini (atau ringkasannya) sbg lampiran metodologi tesis.")

## Kesimpulan Tahap 6 (Pelaporan akhir) (isi setelah eksekusi)

**Tanggal eksekusi**: _(isi)_
**Seed / konfigurasi**: _(isi)_

**Ringkasan hasil**:
_(isi — angka kunci dari sel di atas)_

**Status gerbang**: ✅ Lulus / ⚠ Lulus dengan catatan / ❌ Tidak lulus

**Keputusan / langkah berikut**:
_(isi)_
